# CEHARPS 07 — ตรวจผลและส่งออก / Final validation and export

ตรวจว่าไฟล์หลักเปิดอ่านได้ สร้าง SHA-256 เพื่อการตรวจสอบย้อนกลับ และบีบอัดผลลัพธ์ทั้งหมด


In [ ]:
import hashlib  # TH: นำเข้าเครื่องมือคำนวณ SHA-256 | EN: Import SHA-256 utilities.
import json  # TH: นำเข้าเครื่องมือ JSON | EN: Import JSON utilities.
import shutil  # TH: นำเข้าเครื่องมือสร้าง ZIP | EN: Import archive utilities.
from pathlib import Path  # TH: นำเข้าคลาสจัดการพาธ | EN: Import the path-management class.
import pandas as pd  # TH: นำเข้า pandas สำหรับตรวจ CSV | EN: Import pandas for CSV checks.
from google.colab import drive  # TH: นำเข้าเครื่องมือเชื่อม Drive | EN: Import the Drive connector.
drive.mount("/content/drive")  # TH: เชื่อม Google Drive | EN: Mount Google Drive.
PROJECT_ROOT = Path("/content/drive/MyDrive/CEHARPS")  # TH: กำหนดโฟลเดอร์โครงการ | EN: Define the project folder.
CONFIG = json.loads((PROJECT_ROOT / "config.json").read_text(encoding="utf-8"))  # TH: อ่านค่ากลาง | EN: Load shared settings.

def digest(path: Path) -> str:  # TH: สร้างฟังก์ชันคำนวณแฮชไฟล์ | EN: Define a file-digest function.
    value = hashlib.sha256()  # TH: สร้างตัวสะสม SHA-256 | EN: Create a SHA-256 accumulator.
    with path.open("rb") as stream:  # TH: เปิดไฟล์แบบไบนารี | EN: Open the file in binary mode.
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):  # TH: อ่านไฟล์ทีละ 1 MB | EN: Read the file in 1 MB chunks.
            value.update(chunk)  # TH: เพิ่มข้อมูลเข้าแฮช | EN: Add the chunk to the digest.
    return value.hexdigest()  # TH: คืนค่าแฮช | EN: Return the digest.


In [ ]:
required = [PROJECT_ROOT / "data/segmentation/magset2_sample/manifest.csv", PROJECT_ROOT / "data/processed/health_metadata.json", PROJECT_ROOT / "artifacts/health/selected_health_model.joblib", PROJECT_ROOT / "artifacts/health/health_model_metrics.csv", PROJECT_ROOT / "artifacts/health/health_uncertainty.csv", PROJECT_ROOT / "artifacts/segmentation/unet_best.pt", PROJECT_ROOT / "artifacts/segmentation/deeplabv3plus_best.pt", PROJECT_ROOT / "artifacts/restoration_selection.csv", PROJECT_ROOT / "artifacts/rag/knowledge.faiss", PROJECT_ROOT / "artifacts/rag/chunks.json", PROJECT_ROOT / "artifacts/rag/index_metadata.json", PROJECT_ROOT / "artifacts/rag/retrieval_smoke_test.json"]  # TH: กำหนดไฟล์หลักรวมดัชนี RAG ที่ต้องมี | EN: Define required outputs including the RAG index.
missing = [str(path) for path in required if not path.exists() or path.stat().st_size == 0]  # TH: หาไฟล์ที่หายหรือว่าง | EN: Find missing or empty files.
if missing:  # TH: ตรวจว่าพบไฟล์ผิดปกติหรือไม่ | EN: Check for missing artifacts.
    raise FileNotFoundError("Missing artifacts:\n" + "\n".join(missing))  # TH: หยุดและแสดงรายการไฟล์ | EN: Stop and list missing artifacts.
metrics = pd.read_csv(PROJECT_ROOT / "artifacts/health/health_model_metrics.csv")  # TH: อ่านตัวชี้วัดโมเดลสุขภาพ | EN: Load health-model metrics.
if metrics[["MAE", "RMSE", "R2"]].isna().any().any():  # TH: ตรวจค่าตัวชี้วัดว่าง | EN: Check for missing metric values.
    raise ValueError("Health metrics contain NaN")  # TH: หยุดเมื่อผลประเมินไม่สมบูรณ์ | EN: Stop on incomplete metrics.
manifest_rows = []  # TH: เตรียมรายการไฟล์และแฮช | EN: Initialize artifact manifest rows.
for path in sorted((PROJECT_ROOT / "artifacts").rglob("*")):  # TH: วนตรวจไฟล์ผลลัพธ์ทั้งหมด | EN: Iterate through all artifact files.
    if path.is_file():  # TH: เลือกเฉพาะไฟล์ | EN: Keep files only.
        manifest_rows.append({"path": str(path.relative_to(PROJECT_ROOT)), "bytes": path.stat().st_size, "sha256": digest(path)})  # TH: บันทึกพาธ ขนาด และแฮช | EN: Record path, size, and hash.
manifest = pd.DataFrame(manifest_rows)  # TH: สร้างตาราง manifest | EN: Build the artifact manifest.
manifest.to_csv(PROJECT_ROOT / "artifact_manifest.csv", index=False)  # TH: บันทึก manifest | EN: Save the artifact manifest.
summary = {"health_mode": CONFIG["health_mode"], "artifact_count": len(manifest), "validation": "passed", "warning": "demo mode outputs are not competition evidence" if CONFIG["health_mode"] == "demo" else "review real-data provenance before use"}  # TH: สร้างสรุปสถานะการรัน | EN: Build the run summary.
(PROJECT_ROOT / "run_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")  # TH: บันทึกสรุปการรัน | EN: Save the run summary.
archive = shutil.make_archive(str(PROJECT_ROOT / "CEHARPS_artifacts"), "zip", root_dir=PROJECT_ROOT / "artifacts")  # TH: บีบอัดผลลัพธ์ทั้งหมด | EN: Compress all artifacts.
print(summary)  # TH: แสดงผลตรวจสุดท้าย | EN: Display final validation status.
print("ZIP:", archive)  # TH: แสดงตำแหน่ง ZIP | EN: Display the ZIP path.
